#### Autoloader

In [0]:
USE CATALOG delta_catalog;
CREATE SCHEMA demo_schemaautoloader

In [0]:
CREATE TABLE delta_catalog.demo_schemaautoloader.bankchurned_data
USING DELTA
LOCATION 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/DemoCopyinto/schema/CREATE TABLE delta_catalog.demo_schemaautoloader.bankchurned_data/'
AS
SELECT 
  CAST(Customerid AS INT) AS Customerid,
  Surname,
  CAST(Creditscore AS INT) AS Creditscore,
  Geography,
  Gender,
  CAST(Age AS INT) AS Age,
  CAST(Tenure AS INT) AS Tenure,
  CAST(Balance AS DOUBLE) AS Balance,
  CAST(Estimatedsalary AS DOUBLE) AS Estimatedsalary
FROM read_files(
  'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn/bankchurn_1_50.csv',
  format => 'csv',
  header => 'true'
);

In [0]:
drop table if exists delta_catalog.demo_schemaautoloader.bankchurned_data

In [0]:
%python
#configs
input_path = "abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn/"
checkpoint_path = "abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/checkpoints_autoloader/csv_autoloader/"
schema_location = "abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/schema_autoloader/csv_autoloader/"
target_table = "delta_catalog.demo_schemaautoloader.bankchurned_data"

In [0]:
%python
#read stream (Autoloader)

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .load(input_path)
)

# Write stream to Delta table
(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)  # batch-style execution
    .toTable(target_table)
)

In [0]:
select * from delta_catalog.demo_schemaautoloader.bankchurned_data

In [0]:
select * from (select Customerid, count(*) as cnt from delta_catalog.demo_schemaautoloader.bankchurned_data group by Customerid) where cnt > 1

#### Read CSV from ABFSS and Write to Table with mergeSchema



In [0]:
%python
# Read streamand config same as above only mergeschema is added below
# Write stream to Delta table
(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)  # batch-style execution
    .toTable(target_table)
)


A. Schema Evolution = "addNewColumns" (Recommended)



In [0]:
%python
from pyspark.sql.functions import *

# Configs
input_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/<path>/"
checkpoint_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/checkpoints/csv_autoloader/"
schema_location = "abfss://<container>@<storage-account>.dfs.core.windows.net/schema/csv_autoloader/"
target_table = "catalog.schema.target_table"

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(input_path)
)

write_stream = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
)

write_stream.toTable(target_table)

B. Schema Evolution = "rescue" (Capture unexpected columns)

In [0]:
%python
from pyspark.sql.functions import *
# Configs
input_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/<path>/"
checkpoint_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/checkpoints/csv_autoloader/"
schema_location = "abfss://<container>@<storage-account>.dfs.core.windows.net/schema/csv_autoloader/"
target_table = "catalog.schema.target_table"

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load(input_path)
)

write_stream = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
)

write_stream.toTable(target_table)

C. Schema Evolution = "failOnNewColumns" (Strict mode)

In [0]:
%python
from pyspark.sql.types import *
# Configs
input_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/<path>/"
checkpoint_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/checkpoints/csv_autoloader/"
schema_location = "abfss://<container>@<storage-account>.dfs.core.windows.net/schema/csv_autoloader/"
target_table = "catalog.schema.target_table"

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("amount", DoubleType(), True)
])

df = (
    spark.readStream
    .schema(schema)
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .load(input_path)
)

(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .toTable(target_table)
)

D. Schema Evolution = "none" (Fixed schema)

In [0]:
%python
from pyspark.sql.types import *
# Configs
input_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/<path>/"
checkpoint_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/checkpoints/csv_autoloader/"
schema_location = "abfss://<container>@<storage-account>.dfs.core.windows.net/schema/csv_autoloader/"
target_table = "catalog.schema.target_table"

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("amount", DoubleType(), True)
])

df = (
    spark.readStream
    .schema(schema)
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .load(input_path)
)

(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .toTable(target_table)
)